# 02 자르기

Document 에 본문과 출처를 싣고, 마크다운 헤더로 자른 뒤 토큰 400 / 겹침 80으로 다시 자른다.


In [ ]:
from pathlib import Path  # 경로를 문자열 대신 객체로 다룬다
import os  # 환경변수(OPENAI_API_KEY)를 넣기 위해 쓴다

# 수업 코드는 키가 이미 있는 상태를 가정한다. 이 노트북은 .env를 직접 읽는다.
for _env in (Path("../.env"), Path("../../c3-api/.env")):  # 프로젝트 루트, 옆 폴더 순으로 찾는다
    if not _env.is_file():  # 파일이 없으면 다음 후보
        continue  # 있는 파일만 읽는다
    for _line in _env.read_text(encoding="utf-8").splitlines():  # .env를 한 줄씩
        _line = _line.strip()  # 앞뒤 공백 제거
        if not _line or _line.startswith("#") or "=" not in _line:  # 빈 줄·주석·형식 아닌 줄
            continue  # 건너뛴다
        _k, _v = _line.split("=", 1)  # KEY=VALUE 로 나눈다
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))  # 이미 있으면 덮지 않는다


In [ ]:
from pathlib import Path  # 윈도우/리눅스 모두에서 같은 방식으로 폴더를 가리킨다

CORPUS = Path("../corpus")  # day02 에서 한 단계 위 = 프로젝트의 corpus (day01/corpus 링크)
if not (CORPUS / "text").is_dir():  # VS Code가 루트에서 실행하면 ../corpus 가 빗나간다
    _here = Path.cwd().resolve()  # 지금 작업 폴더
    for _p in [_here, *_here.parents]:  # 위로 올라가며 찾는다
        if (_p / "corpus" / "text").is_dir():  # corpus/text 가 있으면 그게 코퍼스다
            CORPUS = _p / "corpus"  # 찾은 경로로 고친다
            break  # 더 위는 보지 않는다
TEXT = CORPUS / "text"  # 어제 뽑아 둔 글자 파일들이 있는 폴더
print("CORPUS", CORPUS.resolve())  # 실제로 어디를 보는지 확인한다

import json  # manifest.json 과 chunks.jsonl 을 읽고 쓰기 위해 쓴다
from langchain_core.documents import Document  # 본문 + metadata 를 한 객체로 묶는 LangChain 규격


Document 는 `page_content`(글)와 `metadata`(출처 같은 부가정보) 두 칸이다.


In [ ]:
# 문서 객체 만들기
# 본문, 메타데이터
Document(page_content="내용", metadata={"key": "value"})  # 모양만 보여 주는 한 줄. 변수에 안 담는다

manifest_dict = json.loads((CORPUS / "manifest.json").read_text(encoding="utf-8"))  # 어제 장부를 리스트로 읽는다
manifest = dict()  # id 로 바로 찾기 위해 빈 사전을 만든다
for item in manifest_dict:  # 장부 한 줄씩
    manifest[item["id"]] = item  # id 가 열쇠, 나머지 정보가 값

docs = []  # LangChain Document 를 모아 둘 리스트
for i, e in manifest.items():  # i = doc_id, e = 장부 한 줄
    docs.append(  # 리스트에 문서를 추가한다
        Document(  # 본문과 출처를 한 객체로
            page_content=(TEXT / e["text_file"]).read_text(encoding="utf-8"),  # text/ 아래 글자 파일
            metadata={"doc_id": i, "source": e["source"], "type": e["type"]},  # 나중에 필터·출처 표시에 쓴다
        )
    )
print("문서", len(docs), [d.metadata["doc_id"] for d in docs])  # 다섯 개가 나와야 한다


In [ ]:
# 자름 Chunking
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter  # 1.0 이후 갈라진 패키지

# 헤더 기준으로 자르고 싶을 때
headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]  # # 을 h1, ## 을 h2, ### 을 h3 로 metadata 에 남긴다

# 마크다운 구조 기준으로
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers,  # 위에서 정한 헤더만 자른다
    strip_headers=False,  # 헤더 줄을 본문에서 지우지 않는다. 조각만 봐도 제목이 남게
)

# 토큰 수 400에서 80자 겹침 기준으로
size_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",  # 글자 수가 아니라 모델이 세는 토큰으로 자른다
    chunk_size=400,  # 한 조각의 목표 크기
    chunk_overlap=80,  # 경계에서 문장이 잘리지 않게 겹친다
)


In [ ]:
# 구조적으로 자르기
staged = []  # 헤더로 가른 중간 결과
for doc in docs:  # 원본 문서 다섯 개
    if doc.metadata["type"] == "md":  # 마크다운만 헤더로 자른다
        pieces = md_splitter.split_text(doc.page_content)  # 본문을 헤더 단위 Document 리스트로
        for peace in pieces:  # 수업 코드 그대로. 조각(piece) 한 개
            peace.metadata = {**doc.metadata, **peace.metadata}  # 문서 출처 + 헤더(h1/h2/h3)를 합친다
        staged.extend(pieces)  # 중간 리스트에 이어 붙인다
    else:  # 마크다운이 아니면 그냥 통과
        staged.append(doc)  # PDF·CSV 추출문은 헤더가 없으니 통째로 둔다

print(len(staged))  # 구조로 가른 개수. 수업에서는 142
chunks = size_splitter.split_documents(staged)  # 토큰 수 기준으로 다시 자름
print(len(chunks))  # 수업에서는 279 개의 조각
# Doc 5(일반 문서) -> 구조화 자르기 -> 사이즈 자르기


In [ ]:
seq = {}  # 문서별로 몇 번째 조각인지 세기 위한 카운터
for c in chunks:  # 모든 조각
    k = c.metadata["doc_id"]  # 이 조각이 어느 문서에서 왔는지
    seq[k] = seq.get(k, 0)  # 아직 없으면 0부터
    c.metadata["chunk_id"] = f"{k}-{seq[k]:04d}"  # pipa-0000 같은 안정된 이름
    seq[k] += 1  # 다음 조각은 번호가 하나 커진다

print(f"→ 조각 {len(chunks):,}개")  # 전체 개수
for k, n in seq.items():  # 문서별 개수
    print(f"   {k:20s} {n:>4}")  # 어느 문서가 조각을 많이 냈는지 본다


### 저장


In [ ]:
OUT = Path("chunks.jsonl")  # day02 폴더에 한 줄에 조각 하나
with OUT.open("w", encoding="utf-8") as f:  # 덮어쓰기
    for c in chunks:  # 조각마다
        f.write(  # 한 줄 추가
            json.dumps(  # JSON 문자열로
                {"text": c.page_content, "metadata": c.metadata},  # 본문과 출처·헤더·chunk_id
                ensure_ascii=False,  # 한글을 \uXXXX 로 이스케이프하지 않는다
            )
            + "\n"  # jsonl 은 줄바꿈이 레코드 경계다
        )
print("저장", OUT.resolve())  # 파일이 어디에 생겼는지
